# Road Segmentation — Multi-Model Training & Comparison

This notebook trains **all model configurations** automatically and produces a comparison dashboard to help select the best architecture.

### What it does
1. Installs dependencies, clones the repo, downloads the dataset
2. Runs training for each config in `configs/` (U-Net, DeepLabV3+, LinkNet, SegFormer)
3. Collects all results and produces:
   - Side-by-side metrics comparison table
   - Overlaid training curves (loss, IoU)
   - Prediction comparison grid across all models
   - Final recommendation
4. Saves all checkpoints and results to Google Drive (optional)

### How to run on Colab
1. `Runtime > Change runtime type > T4 GPU`
2. Set `REPO_URL` with your GitHub token (for private repos)
3. Set Kaggle credentials in the dataset cell
4. **Run All** (`Runtime > Run all`) — everything is automated

---
## 1. Setup & Installation

In [ ]:
import subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules

# ====== SET YOUR REPO URL (use token for private repos) ======
REPO_URL = "https://github.com/minaessam2015/road-segmentation.git"
# REPO_URL = "https://<TOKEN>@github.com/minaessam2015/road-segmentation.git"

if IN_COLAB:
    REPO_DIR = "/content/road-segmentation"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
        print(f"Cloned to {REPO_DIR}")
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
        print(f"Pulled latest to {REPO_DIR}")

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub", "pyyaml", "tqdm"], check=True)
    print("Dependencies installed.")
else:
    REPO_DIR = None
    print("Running locally.")

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

if IN_COLAB:
    PROJECT_ROOT = Path(REPO_DIR).resolve()
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

---
## 2. Device Detection

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple MPS")
else:
    device = torch.device("cpu")
    print("Using CPU")

print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

---
## 3. Download Dataset

In [ ]:
from road_segmentation.paths import DEEPGLOBE_DATASET_DIR

train_dir = DEEPGLOBE_DATASET_DIR / "train"
has_images = train_dir.exists() and any(train_dir.glob("*_sat.*"))

if not has_images:
    if IN_COLAB:
        kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
        if not kaggle_json.exists():
            # Option A: Upload kaggle.json
            print("Upload your kaggle.json file:")
            from google.colab import files
            uploaded = files.upload()
            kaggle_json.parent.mkdir(parents=True, exist_ok=True)
            kaggle_json.write_bytes(list(uploaded.values())[0])
            kaggle_json.chmod(0o600)

        # Option B: Set credentials directly (uncomment):
        # os.environ["KAGGLE_USERNAME"] = "your_username"
        # os.environ["KAGGLE_KEY"] = "your_api_key"

    print("Downloading DeepGlobe dataset...")
    from road_segmentation.data.download import download_dataset
    download_dataset()

sat_count = len(list((DEEPGLOBE_DATASET_DIR / "train").glob("*_sat.*")))
mask_count = len(list((DEEPGLOBE_DATASET_DIR / "train").glob("*_mask.*")))
print(f"Training: {sat_count} images, {mask_count} masks")

---
## 4. Experiment Settings

Choose which configs to run and set shared overrides (subset size, epochs, etc.).

In [ ]:
# ====== CONFIGS TO COMPARE ======
# Comment out any you want to skip
CONFIGS_TO_RUN = [
    "unet_resnet34.yaml",
    "deeplabv3plus_resnet50.yaml",
    "linknet_resnet34.yaml",
    # "segformer_mitb2.yaml",  # uncomment if you have enough GPU memory
]

# ====== SHARED OVERRIDES ======
# These apply to ALL experiments for a fair comparison
SHARED_OVERRIDES = {
    # "data.subset_size": 1000,   # use a subset for quick experiments
    # "training.epochs": 15,      # reduce epochs for speed
}

# Auto-adjust for environment
if device.type == "cpu":
    SHARED_OVERRIDES.setdefault("data.subset_size", 200)
    SHARED_OVERRIDES.setdefault("training.epochs", 5)
    SHARED_OVERRIDES.setdefault("data.num_workers", 0)
    SHARED_OVERRIDES.setdefault("data.batch_size", 4)
    SHARED_OVERRIDES.setdefault("training.mixed_precision", False)

if IN_COLAB:
    SHARED_OVERRIDES.setdefault("data.num_workers", 2)

print(f"Experiments to run: {len(CONFIGS_TO_RUN)}")
for c in CONFIGS_TO_RUN:
    print(f"  - {c}")
if SHARED_OVERRIDES:
    print(f"\nShared overrides:")
    for k, v in SHARED_OVERRIDES.items():
        print(f"  {k} = {v}")

---
## 5. Run All Experiments

This cell loops through each config, trains the model, and collects results.

In [ ]:
import gc
import random
import time

import numpy as np
import pandas as pd

from road_segmentation.config import load_config
from road_segmentation.data.eda import discover_image_mask_pairs
from road_segmentation.data.split import split_pairs
from road_segmentation.data.transforms import get_train_transform, get_val_transform
from road_segmentation.data.dataset import create_dataloaders
from road_segmentation.models.factory import create_model, count_parameters, freeze_encoder
from road_segmentation.training.losses import create_loss
from road_segmentation.training.trainer import Trainer

# Collect results across experiments
all_results = {}  # config_name -> {"config", "trainer", "best_metrics", "history", "train_time"}

for config_name in CONFIGS_TO_RUN:
    print("\n" + "=" * 70)
    print(f"  EXPERIMENT: {config_name}")
    print("=" * 70)

    # --- Load config with overrides ---
    config = load_config(PROJECT_ROOT / "configs" / config_name)
    for key, value in SHARED_OVERRIDES.items():
        parts = key.split(".")
        obj = config
        for part in parts[:-1]:
            obj = getattr(obj, part)
        setattr(obj, parts[-1], value)

    # --- Seed ---
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(config.seed)

    # --- Data (same split for all experiments) ---
    pairs = discover_image_mask_pairs(DEEPGLOBE_DATASET_DIR)
    train_pairs, val_pairs = split_pairs(
        pairs,
        val_ratio=config.data.val_split_ratio,
        seed=config.data.val_split_seed,
        subset_size=config.data.subset_size,
    )

    train_transform = get_train_transform(
        image_size=config.data.image_size,
        aug_steps=config.augmentations.train,
        mean=config.normalization.mean,
        std=config.normalization.std,
    )
    val_transform = get_val_transform(
        image_size=config.data.image_size,
        mean=config.normalization.mean,
        std=config.normalization.std,
    )
    train_loader, val_loader = create_dataloaders(
        train_pairs, val_pairs,
        train_transform, val_transform,
        batch_size=config.data.batch_size,
        num_workers=config.data.num_workers,
        pin_memory=config.data.pin_memory and device.type == "cuda",
    )

    # --- Model ---
    model = create_model(
        arch=config.model.arch,
        encoder_name=config.model.encoder_name,
        encoder_weights=config.model.encoder_weights,
        in_channels=config.model.in_channels,
        classes=config.model.classes,
        **config.model.decoder_kwargs,
    )
    if config.training.freeze_encoder_epochs > 0:
        freeze_encoder(model)

    params = count_parameters(model)
    print(f"Model: {config.model.arch} + {config.model.encoder_name} | {params['total']/1e6:.1f}M params")

    # --- Train ---
    loss_fn = create_loss(config.loss.type, config.loss.params)
    trainer = Trainer(config, model, train_loader, val_loader, loss_fn, device)

    t0 = time.time()
    best_metrics = trainer.train()
    train_time = time.time() - t0

    # --- Collect results ---
    label = f"{config.model.arch}+{config.model.encoder_name}"
    all_results[label] = {
        "config_name": config_name,
        "config": config,
        "trainer": trainer,
        "best_metrics": best_metrics,
        "history": trainer.history,
        "train_time_min": train_time / 60,
        "total_params_m": params["total"] / 1e6,
    }

    print(f"\nDone: val_iou={best_metrics.get('val_iou', 0):.4f} | time={train_time/60:.1f} min")

    # Free GPU memory before next experiment
    del model, trainer, loss_fn, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 70)
print(f"  ALL {len(all_results)} EXPERIMENTS COMPLETE")
print("=" * 70)

---
## 6. Comparison Dashboard

### 6a. Metrics Comparison Table

In [ ]:
import pandas as pd

rows = []
for label, result in all_results.items():
    bm = result["best_metrics"]
    rows.append({
        "Model": label,
        "Best IoU": bm.get("val_iou", 0),
        "Best Dice": bm.get("val_dice", 0),
        "Precision": bm.get("val_precision", 0),
        "Recall": bm.get("val_recall", 0),
        "Val Loss": bm.get("val_loss", 0),
        "Params (M)": result["total_params_m"],
        "Time (min)": result["train_time_min"],
    })

comparison_df = pd.DataFrame(rows).sort_values("Best IoU", ascending=False)
comparison_df = comparison_df.reset_index(drop=True)
comparison_df.index += 1  # rank starting from 1
comparison_df.index.name = "Rank"

print("Model Comparison (ranked by IoU)")
print("=" * 70)
display(comparison_df.round(4))

winner = comparison_df.iloc[0]["Model"]
print(f"\nBest model: {winner} (IoU={comparison_df.iloc[0]['Best IoU']:.4f})")

### 6b. Training Curves — All Models Overlaid

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("ggplot")
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

for label, result in all_results.items():
    history = result["history"]
    epochs = [h["epoch"] for h in history]

    axes[0, 0].plot(epochs, [h["train_loss"] for h in history], label=f"{label} (train)", linewidth=1.5)
    axes[0, 0].plot(epochs, [h["val_loss"] for h in history], label=f"{label} (val)", linestyle="--", linewidth=1.5)

    axes[0, 1].plot(epochs, [h["val_iou"] for h in history], label=label, linewidth=2)

    axes[1, 0].plot(epochs, [h["val_dice"] for h in history], label=label, linewidth=2)

    axes[1, 1].plot(epochs, [h["lr"] for h in history], label=label, linewidth=1.5)

axes[0, 0].set_title("Loss", fontsize=13)
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(True)

axes[0, 1].set_title("Validation IoU", fontsize=13)
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True)

axes[1, 0].set_title("Validation Dice", fontsize=13)
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].legend(fontsize=9)
axes[1, 0].grid(True)

axes[1, 1].set_title("Learning Rate", fontsize=13)
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_yscale("log")
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True)

fig.suptitle("Training Curves — All Models", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

### 6c. Prediction Comparison — Same Samples Across All Models

In [ ]:
from road_segmentation.training.visualization import denormalize_image

# Load best checkpoint for each model and predict on the same val samples
N_SAMPLES = 4
n_models = len(all_results)

# Get a fixed val batch from the last experiment's config
last_config = list(all_results.values())[-1]["config"]
pairs = discover_image_mask_pairs(DEEPGLOBE_DATASET_DIR)
_, val_pairs = split_pairs(
    pairs,
    val_ratio=last_config.data.val_split_ratio,
    seed=last_config.data.val_split_seed,
    subset_size=last_config.data.subset_size,
)
val_transform = get_val_transform(
    image_size=last_config.data.image_size,
    mean=last_config.normalization.mean,
    std=last_config.normalization.std,
)
from road_segmentation.data.dataset import RoadSegmentationDataset
val_ds = RoadSegmentationDataset(val_pairs[:N_SAMPLES], transform=val_transform)
samples = [val_ds[i] for i in range(N_SAMPLES)]
images = torch.stack([s["image"] for s in samples])
masks_gt = torch.stack([s["mask"] for s in samples])

# columns: Input | GT | model1 pred | model2 pred | ...
n_cols = 2 + n_models
fig, axes = plt.subplots(N_SAMPLES, n_cols, figsize=(4 * n_cols, 4 * N_SAMPLES))
if N_SAMPLES == 1:
    axes = axes[np.newaxis, :]

mean = last_config.normalization.mean
std = last_config.normalization.std

for i in range(N_SAMPLES):
    # Input image
    img = denormalize_image(images[i], mean, std)
    axes[i, 0].imshow(img)
    axes[i, 0].axis("off")
    if i == 0:
        axes[i, 0].set_title("Input", fontsize=11, fontweight="bold")

    # Ground truth
    axes[i, 1].imshow(masks_gt[i, 0].numpy(), cmap="gray")
    axes[i, 1].axis("off")
    if i == 0:
        axes[i, 1].set_title("Ground Truth", fontsize=11, fontweight="bold")

for col_idx, (label, result) in enumerate(all_results.items()):
    # Load best model
    cfg = result["config"]
    model = create_model(
        arch=cfg.model.arch,
        encoder_name=cfg.model.encoder_name,
        encoder_weights=None,  # load from checkpoint, not imagenet
        in_channels=cfg.model.in_channels,
        classes=cfg.model.classes,
    )
    ckpt_dir = Path(cfg.checkpoint.save_dir) / cfg.logging.experiment_name
    best_ckpt = ckpt_dir / "best.pth"
    if best_ckpt.exists():
        state = torch.load(best_ckpt, map_location="cpu", weights_only=False)
        model.load_state_dict(state["model_state_dict"])
    model.eval().to(device)

    with torch.no_grad():
        logits = model(images.to(device))
        preds = (torch.sigmoid(logits) >= 0.5).cpu().numpy()

    iou = result["best_metrics"].get("val_iou", 0)
    for i in range(N_SAMPLES):
        axes[i, 2 + col_idx].imshow(preds[i, 0], cmap="gray")
        axes[i, 2 + col_idx].axis("off")
        if i == 0:
            axes[i, 2 + col_idx].set_title(f"{label}\nIoU={iou:.3f}", fontsize=10, fontweight="bold")

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

fig.suptitle("Prediction Comparison Across Models", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 6d. Efficiency Plot — IoU vs Training Time vs Model Size

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for label, result in all_results.items():
    iou = result["best_metrics"].get("val_iou", 0)
    time_min = result["train_time_min"]
    params_m = result["total_params_m"]

    # Bubble size proportional to model size
    ax.scatter(time_min, iou, s=params_m * 15, alpha=0.7, edgecolors="black", linewidth=1)
    ax.annotate(label, (time_min, iou), textcoords="offset points",
                xytext=(10, 5), fontsize=9)

ax.set_xlabel("Training Time (minutes)", fontsize=12)
ax.set_ylabel("Best Validation IoU", fontsize=12)
ax.set_title("Efficiency: IoU vs Training Time (bubble size = params)", fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Full Training History

In [ ]:
for label, result in all_results.items():
    print(f"\n{label}")
    print("-" * 50)
    df = pd.DataFrame(result["history"])
    display(df.round(4))

---
## 8. Save Results (Optional)

Save all checkpoints and comparison results to Google Drive.

In [ ]:
import shutil

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    save_dir = Path("/content/drive/MyDrive/road_segmentation")
else:
    save_dir = PROJECT_ROOT / "experiment_results"

save_dir.mkdir(parents=True, exist_ok=True)

# Save comparison table
comparison_df.to_csv(save_dir / "comparison.csv")
print(f"Comparison table saved to: {save_dir / 'comparison.csv'}")

# Save best checkpoint for each model
for label, result in all_results.items():
    cfg = result["config"]
    ckpt_dir = Path(cfg.checkpoint.save_dir) / cfg.logging.experiment_name
    best_ckpt = ckpt_dir / "best.pth"
    if best_ckpt.exists():
        dest = save_dir / f"{label}_best.pth"
        shutil.copy(best_ckpt, dest)
        print(f"  {label} -> {dest}")

print(f"\nAll results saved to: {save_dir}")